# 04 Uncertainty And Reference Sensitivity

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roymustang11/InSAR-Benchmark-Lab/blob/main/notebooks/04_uncertainty_and_reference_sensitivity.ipynb)

This notebook demonstrates how reference-area choice, masking thresholds, and uncertainty assumptions can change validation results.

**Important:** `DEMONSTRATION_DATA = True`. This is not a real deformation result. The arrays are deterministic fixtures designed to exercise the benchmark logic before real OPERA or MintPy products are connected.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/roymustang11/InSAR-Benchmark-Lab.git"
DEMONSTRATION_DATA = True

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    project_root = Path("/content/InSAR-Benchmark-Lab")
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(project_root)])
else:
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Using demonstration fixture data: {DEMONSTRATION_DATA}")


## Why This Matters

InSAR displacement is relative. Reference-area selection, masking thresholds, and uncertainty assumptions can change estimated velocities and validation residuals. A professional benchmark should report those sensitivities instead of hiding them in preprocessing choices.


In [ ]:
import numpy as np
import pandas as pd

from insar_benchmark_lab.metrics import rmse, trend_bias, uncertainty_coverage

rng = np.random.default_rng(42)
n_points = 48
true_velocity_mm_per_year = np.linspace(-42, -8, n_points)
gnss_velocity_mm_per_year = true_velocity_mm_per_year + rng.normal(0, 1.2, n_points)
coherence = np.linspace(0.32, 0.92, n_points)
base_insar_velocity = true_velocity_mm_per_year + rng.normal(0, 2.0, n_points)
sigma_velocity = np.interp(coherence, [0.32, 0.92], [6.0, 1.8])

reference_candidates = {
    "stable_west": 0.0,
    "slightly_subsiding_center": -3.5,
    "noisy_east": 2.2,
}
mask_thresholds = [0.35, 0.5, 0.65, 0.8]

print("Reference candidates:", reference_candidates)
print("Mask thresholds:", mask_thresholds)


## Reference And Mask Sensitivity Matrix

Each row below represents one analysis choice: a reference candidate and a coherence mask threshold. Real notebooks should compute the same table after loading product-derived velocities.


In [ ]:
rows = []
for reference_name, reference_shift in reference_candidates.items():
    referenced_insar = base_insar_velocity - reference_shift
    for threshold in mask_thresholds:
        mask = coherence >= threshold
        rows.append(
            {
                "reference_name": reference_name,
                "reference_shift_mm_per_year": reference_shift,
                "mask_threshold": threshold,
                "n_points": int(mask.sum()),
                "rmse_mm_per_year": rmse(gnss_velocity_mm_per_year[mask], referenced_insar[mask]),
                "bias_mm_per_year": trend_bias(gnss_velocity_mm_per_year[mask], referenced_insar[mask]),
                "one_sigma_coverage": uncertainty_coverage(
                    gnss_velocity_mm_per_year[mask],
                    referenced_insar[mask],
                    sigma_velocity[mask],
                ),
            }
        )

sensitivity_table = pd.DataFrame(rows)
sensitivity_table


In [ ]:
import matplotlib.pyplot as plt

figure_panels, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)

for reference_name, group in sensitivity_table.groupby("reference_name"):
    axes[0].plot(group["mask_threshold"], group["rmse_mm_per_year"], marker="o", label=reference_name)
    axes[1].plot(group["mask_threshold"], group["bias_mm_per_year"], marker="o", label=reference_name)
    axes[2].plot(group["mask_threshold"], group["one_sigma_coverage"], marker="o", label=reference_name)

axes[0].set_title("RMSE sensitivity")
axes[0].set_ylabel("mm/year")
axes[1].set_title("Bias sensitivity")
axes[1].axhline(0, color="black", linewidth=1)
axes[2].set_title("1-sigma coverage")
axes[2].axhline(0.68, color="black", linestyle="--", linewidth=1, label="ideal Gaussian 1-sigma")

for ax in axes:
    ax.set_xlabel("Coherence mask threshold")
    ax.grid(True, alpha=0.3)

axes[2].legend(loc="best", fontsize=8)
figure_panels.suptitle("Demonstration Reference And Mask Sensitivity")
plt.show()


## Interpretation Template

For real data, report the best-performing choice and the spread across reasonable choices. A defensible InSAR result should describe how much velocity, bias, and uncertainty coverage change when reference and mask assumptions are perturbed.
